# PLM Subcellular Localization — 262 Curated Sequences Only

**Per Prof. Gruber's direction:** Train exclusively on the 262 curated *P. tricornutum* sequences from `data.xlsx`.

**Pipeline:**
1. Setup & install
2. 5-fold cross-validation (homology-aware splits)
3. Train final model on all 262 sequences
4. Predict on test organisms
5. Compare with ASAFind & compute Baldi metrics

**Runtime:** Select **T4 GPU** (Runtime → Change runtime type → T4 GPU)

## Cell 1: Clone repo & install dependencies

In [ ]:
!git clone https://github.com/nkhthorat/subcellular-localization-PLM.git
%cd subcellular-localization-PLM
!pip install -q torch transformers sentencepiece protobuf scikit-learn openpyxl joblib

## Cell 1b: Apply our improvements
These add: predict.py, train_final.py, compare_predictions.py, apply_prior_correction.py, compute_full_metrics.py, and backbone caching in embeddings.py

In [ ]:
%%writefile scripts/train_final.py
#!/usr/bin/env python3
"""Train a final model on all data and save it for prediction."""
from __future__ import annotations

import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))

import argparse
import os

import joblib
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from plm_localization.config import Config
from plm_localization.data import load_dataset
from plm_localization.embeddings import compute_features
from plm_localization.model import PooledMLP
from plm_localization.utils import setup_logger, set_seed


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="data.xlsx")
    p.add_argument("--save-dir", default="runs/final_model")
    p.add_argument("--backbone", choices=["prott5", "esm1b"], default="prott5")
    p.add_argument("--nterm", type=int, default=250)
    p.add_argument("--pooling", default="mean_prefix_50")
    p.add_argument("--seed", type=int, default=200)
    args = p.parse_args()

    logger = setup_logger("INFO")
    os.makedirs(args.save_dir, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    cfg = Config(
        data_path=args.data,
        save_dir=args.save_dir,
        backbone=args.backbone,
        prott5_name="Rostlab/prot_t5_xl_uniref50",
        esm1b_name="facebook/esm1b_t33_650M_UR50S",
        last_k=4,
        nterm_max_res=args.nterm,
        pooling=args.pooling,
        use_homology_aware_cv=True,
        min_seq_id=0.30,
        k_folds=5,
        graphpart_threads=2,
        prefer_graphpart_partitions=True,
        seed=args.seed,
        lr=1e-3,
        weight_decay=1e-5,
        h1=512,
        h2=128,
        dropout=0.3,
        label_smoothing=0.05,
        batch_size=64,
        epochs=50,
        early_stop_patience=8,
        force_deterministic=False,
        device=device,
        fp16=(device == "cuda"),
        embed_batch_size=8,
        cache_embeddings=True,
    )

    set_seed(cfg.seed, cfg.force_deterministic)

    df, le = load_dataset(cfg.data_path)
    joblib.dump(le, os.path.join(cfg.save_dir, "label_encoder.pkl"))
    logger.info("Loaded %d sequences, %d classes: %s", len(df), len(le.classes_), list(le.classes_))

    logger.info("Computing embeddings for all %d sequences...", len(df))
    X = compute_features(df["sequence_clean"].astype(str).tolist(), cfg)
    y = df["label_enc"].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    joblib.dump(scaler, os.path.join(cfg.save_dir, "scaler.pkl"))

    tr_ds = TensorDataset(
        torch.tensor(X_scaled, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )
    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True)

    model = PooledMLP(X_scaled.shape[1], len(le.classes_), cfg.h1, cfg.h2, cfg.dropout).to(torch.device(device))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    crit = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)

    logger.info("Training final model on all %d sequences for %d epochs...", len(df), cfg.epochs)
    for ep in range(1, cfg.epochs + 1):
        model.train()
        total_loss = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item()
        if ep % 10 == 0:
            logger.info("Epoch %d/%d  loss=%.4f", ep, cfg.epochs, total_loss / len(tr_loader))

    torch.save(model.state_dict(), os.path.join(cfg.save_dir, "model.pt"))
    logger.info("Saved model to %s", args.save_dir)
    logger.info("Files saved: model.pt, scaler.pkl, label_encoder.pkl")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/predict.py
#!/usr/bin/env python3
"""Predict subcellular localization for new protein sequences."""
from __future__ import annotations

import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))

import argparse
import csv
import os

import joblib
import numpy as np
import torch

from plm_localization.config import Config
from plm_localization.data import clean_and_map
from plm_localization.embeddings import compute_features
from plm_localization.model import PooledMLP
from plm_localization.utils import setup_logger


def load_model(model_dir: str, device: str):
    le = joblib.load(os.path.join(model_dir, "label_encoder.pkl"))
    scaler = joblib.load(os.path.join(model_dir, "scaler.pkl"))
    in_dim = scaler.mean_.shape[0]
    n_classes = len(le.classes_)
    model = PooledMLP(in_dim, n_classes, h1=512, h2=128, dropout=0.0)
    model.load_state_dict(torch.load(os.path.join(model_dir, "model.pt"), map_location=device))
    model.to(device).eval()
    return model, scaler, le


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model-dir", default="runs/final_model")
    p.add_argument("--sequence", type=str, default=None)
    p.add_argument("--fasta", type=str, default=None)
    p.add_argument("--backbone", choices=["prott5", "esm1b"], default="prott5")
    p.add_argument("--nterm", type=int, default=250)
    p.add_argument("--pooling", default="mean_prefix_50")
    p.add_argument("--probs-out", default=None, help="Save full per-class probabilities to CSV")
    args = p.parse_args()

    logger = setup_logger("INFO")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    if args.sequence is None and args.fasta is None:
        p.error("Provide either --sequence or --fasta")

    seq_ids, raw_seqs = [], []
    if args.sequence:
        seq_ids.append("input_sequence")
        raw_seqs.append(args.sequence)
    if args.fasta:
        current_id, current_seq = None, []
        with open(args.fasta) as f:
            for line in f:
                line = line.strip()
                if line.startswith(">"):
                    if current_id is not None:
                        seq_ids.append(current_id)
                        raw_seqs.append("".join(current_seq))
                    current_id = line[1:].split()[0]
                    current_seq = []
                else:
                    current_seq.append(line)
        if current_id is not None:
            seq_ids.append(current_id)
            raw_seqs.append("".join(current_seq))

    clean_seqs = [clean_and_map(s) for s in raw_seqs]

    cfg = Config(
        data_path="", save_dir=args.model_dir,
        backbone=args.backbone,
        prott5_name="Rostlab/prot_t5_xl_uniref50",
        esm1b_name="facebook/esm1b_t33_650M_UR50S",
        last_k=4, nterm_max_res=args.nterm, pooling=args.pooling,
        use_homology_aware_cv=True, min_seq_id=0.30, k_folds=5,
        graphpart_threads=2, prefer_graphpart_partitions=True,
        seed=0, lr=1e-3, weight_decay=1e-5, h1=512, h2=128,
        dropout=0.3, label_smoothing=0.05, batch_size=64, epochs=50,
        early_stop_patience=8, force_deterministic=False,
        device=device, fp16=(device == "cuda"),
        embed_batch_size=8, cache_embeddings=False,
    )

    logger.info("Embedding %d sequence(s)...", len(clean_seqs))
    X = compute_features(clean_seqs, cfg)

    model, scaler, le = load_model(args.model_dir, device)
    X_scaled = scaler.transform(X)

    with torch.no_grad():
        logits = model(torch.tensor(X_scaled, dtype=torch.float32).to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        pred_idx = np.argmax(probs, axis=1)

    print(f"\n{'ID':<30} {'Prediction':<50} {'Confidence'}")
    print("-" * 95)
    for sid, idx, prob_row in zip(seq_ids, pred_idx, probs):
        label = le.classes_[idx]
        confidence = prob_row[idx]
        print(f"{sid:<30} {label:<50} {confidence:.1%}")

    if len(seq_ids) == 1:
        print("\nAll class probabilities:")
        for cls, prob in zip(le.classes_, probs[0]):
            print(f"  {cls:<50} {prob:.1%}")

    if args.probs_out:
        with open(args.probs_out, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["sequence_id"] + list(le.classes_))
            for sid, prob_row in zip(seq_ids, probs):
                writer.writerow([sid] + [f"{p:.6f}" for p in prob_row])
        print(f"\nFull probabilities saved to {args.probs_out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/compare_predictions.py
#!/usr/bin/env python3
"""Compare model predictions against ASAFind 2.0 predictions."""
from __future__ import annotations
import argparse, re, csv

def normalize_asafind(label):
    label = label.strip()
    if label.startswith("Plastid"): return "Plastid"
    if label == "Not plastid, signal peptide identified": return "Not plastid, not ppc, signal peptide identified"
    if "PPC" in label: return "PPC"
    return label

def parse_model_output(path):
    results = {}
    line_re = re.compile(r"^(\S+)\s+(.+?)\s+([\d.]+)%\s*$")
    with open(path) as f:
        for raw in f:
            line = raw.rstrip("\n")
            if not line or line.startswith("ID") or set(line.strip()) == {"-"}: continue
            if line.startswith("All class probabilities"): break
            m = line_re.match(line)
            if not m: continue
            results[m.group(1)] = (m.group(2).strip(), float(m.group(3)))
    return results

def parse_asafind_tab(path):
    results = {}
    with open(path) as f:
        f.readline()
        for raw in f:
            line = raw.rstrip("\n")
            if not line.strip(): continue
            cols = line.split("\t")
            results[cols[0].strip()] = normalize_asafind(cols[-1].strip())
    return results

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model-output", required=True)
    ap.add_argument("--asafind-tab", required=True)
    ap.add_argument("--out", default="comparison.csv")
    args = ap.parse_args()
    model_preds = parse_model_output(args.model_output)
    asafind_preds = parse_asafind_tab(args.asafind_tab)
    print(f"Parsed {len(model_preds)} model predictions")
    print(f"Parsed {len(asafind_preds)} ASAFind predictions")
    all_ids = sorted(set(model_preds) | set(asafind_preds))
    rows, n_matched, n_agree = [], 0, 0
    for seq_id in all_ids:
        model_label, model_conf = model_preds.get(seq_id, ("MISSING", None))
        asafind_label = asafind_preds.get(seq_id, "MISSING")
        matched = model_label != "MISSING" and asafind_label != "MISSING"
        agree = matched and (model_label == asafind_label)
        if matched: n_matched += 1
        if agree: n_agree += 1
        rows.append({"sequence_id": seq_id, "model_prediction": model_label,
                      "model_confidence": model_conf, "asafind_prediction": asafind_label,
                      "agree": agree if matched else "N/A"})
    with open(args.out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    print(f"\nWrote {len(rows)} rows to {args.out}")
    print(f"Sequences in both: {n_matched}")
    if n_matched: print(f"Agreement: {n_agree}/{n_matched} ({100*n_agree/n_matched:.1f}%)")
    print("\nDisagreements (model vs ASAFind):")
    dc = {}
    for r in rows:
        if r["agree"] is False:
            k = (r["model_prediction"], r["asafind_prediction"])
            dc[k] = dc.get(k, 0) + 1
    for (m, a), c in sorted(dc.items(), key=lambda x: -x[1]):
        print(f"  model='{m}' vs asafind='{a}'  x{c}")

if __name__ == "__main__": main()

In [ ]:
%%writefile scripts/compute_full_metrics.py
#!/usr/bin/env python3
"""Compute full Baldi-style evaluation metrics from comparison CSV."""
from __future__ import annotations
import argparse, csv, sys
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, matthews_corrcoef

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True)
    ap.add_argument("--pred-col", default="model_prediction")
    ap.add_argument("--true-col", default="asafind_prediction")
    args = ap.parse_args()
    y_true, y_pred = [], []
    with open(args.csv) as f:
        for row in csv.DictReader(f):
            if row[args.pred_col] == "MISSING" or row[args.true_col] == "MISSING": continue
            y_true.append(row[args.true_col]); y_pred.append(row[args.pred_col])
    if not y_true: print("No matched sequences."); sys.exit(1)
    labels = sorted(set(y_true) | set(y_pred))
    print(f"Matched sequences: {len(y_true)}")
    print(f"Labels: {labels}\n")
    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.4f} ({acc*100:.1f}%)")
    print(f"  (Note: accuracy can be misleading under class imbalance)\n")
    mcc = matthews_corrcoef(y_true, y_pred)
    print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}\n")
    print(f"Macro-F1:    {f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0):.4f}")
    print(f"Weighted-F1: {f1_score(y_true, y_pred, labels=labels, average='weighted', zero_division=0):.4f}\n")
    print("Per-class classification report:")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print("Confusion Matrix (rows=true, cols=predicted):")
    print(f"{'':>55s} " + " ".join(f"{l[:12]:>12s}" for l in labels))
    for i, label in enumerate(labels):
        print(f"{label:>55s} " + " ".join(f"{cm[i,j]:12d}" for j in range(len(labels))))

if __name__ == "__main__": main()

In [ ]:
%%writefile scripts/apply_prior_correction.py
#!/usr/bin/env python3
"""Apply prior (class-imbalance) correction to model predictions."""
from __future__ import annotations
import argparse, csv, math
from collections import Counter

TRAIN_COUNTS = {"Plastid": 71, "PPC": 44,
    "Not plastid, not ppc, signal peptide identified": 92,
    "TargetP-2.0: mTP": 26, "TargetP-2.0: noTP": 29}

def normalize_asafind(label):
    label = label.strip()
    if label.startswith("Plastid"): return "Plastid"
    if label == "Not plastid, signal peptide identified": return "Not plastid, not ppc, signal peptide identified"
    if "PPC" in label: return "PPC"
    return label

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--probs", required=True)
    ap.add_argument("--asafind-tab", default=None)
    ap.add_argument("--target-prior-from-asafind", action="store_true")
    ap.add_argument("--out", default="corrected_predictions.csv")
    args = ap.parse_args()
    # Load probs
    with open(args.probs) as f:
        reader = csv.reader(f); header = next(reader)
        classes = header[1:]
        probs_data = {}
        for row in reader:
            probs_data[row[0]] = {c: float(v) for c, v in zip(classes, row[1:])}
    print(f"Loaded {len(probs_data)} sequences, classes: {classes}")
    train_total = sum(TRAIN_COUNTS.values())
    train_prior = {c: TRAIN_COUNTS[c]/train_total for c in classes}
    asafind_preds = None
    if args.asafind_tab:
        asafind_preds = {}
        with open(args.asafind_tab) as f:
            f.readline()
            for raw in f:
                line = raw.rstrip("\n")
                if not line.strip(): continue
                cols = line.split("\t")
                asafind_preds[cols[0].strip()] = normalize_asafind(cols[-1].strip())
        print(f"Loaded {len(asafind_preds)} ASAFind predictions")
    if args.target_prior_from_asafind:
        if not asafind_preds: raise SystemExit("Need --asafind-tab")
        counts = Counter(asafind_preds.values())
        total = sum(counts.values())
        target_prior = {c: max(counts.get(c,0),1)/total for c in classes}
        print("\nEstimated target prior (from ASAFind):")
        for c in classes:
            print(f"  {c:55s} train={train_prior[c]:.3f}  target={target_prior[c]:.3f}")
    else:
        target_prior = train_prior
    log_adjust = {c: math.log(target_prior[c]) - math.log(train_prior[c]) for c in classes}
    rows, n_changed = [], 0
    for sid, probs in probs_data.items():
        orig = max(probs, key=probs.get)
        adjusted = {c: math.log(max(probs[c],1e-12)) + log_adjust[c] for c in classes}
        corrected = max(adjusted, key=adjusted.get)
        if corrected != orig: n_changed += 1
        rows.append({"sequence_id": sid, "original_prediction": orig,
                      "corrected_prediction": corrected, "changed": corrected != orig})
    with open(args.out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["sequence_id","original_prediction","corrected_prediction","changed"])
        w.writeheader(); w.writerows(rows)
    print(f"\nWrote {len(rows)} rows to {args.out}")
    print(f"Changed: {n_changed}/{len(rows)} ({100*n_changed/len(rows):.1f}%)")
    if asafind_preds:
        n_m = n_ab = n_aa = 0
        for r in rows:
            if r["sequence_id"] not in asafind_preds: continue
            n_m += 1
            if r["original_prediction"] == asafind_preds[r["sequence_id"]]: n_ab += 1
            if r["corrected_prediction"] == asafind_preds[r["sequence_id"]]: n_aa += 1
        print(f"\n=== Agreement with ASAFind ===")
        print(f"Matched: {n_m}")
        print(f"BEFORE: {n_ab}/{n_m} ({100*n_ab/n_m:.1f}%)")
        print(f"AFTER:  {n_aa}/{n_m} ({100*n_aa/n_m:.1f}%)")

if __name__ == "__main__": main()

## Cell 2: 5-Fold Cross-Validation on 262 sequences
Homology-aware splits (GraphPart at 30% seq identity threshold)

In [ ]:
!python3 scripts/run_experiment.py --data data.xlsx --save-dir runs/cv_262seq

In [ ]:
# View the cross-validation results
!echo "=== OOF Classification Report ==="
!cat runs/cv_262seq/oof_report.txt
!echo "\n=== Confusion Matrix ==="
!cat runs/cv_262seq/oof_confusion_matrix.csv
!echo "\n=== Summary ==="
!python3 -c "import json; d=json.load(open('runs/cv_262seq/summary.json')); print(f'Acc={d[\"acc\"]:.4f}  Macro-F1={d[\"macro_f1\"]:.4f}  Weighted-F1={d[\"weighted_f1\"]:.4f}  MCC={d[\"mcc\"]:.4f}')"

## Cell 3: Train Final Model on all 262 sequences

In [ ]:
!python3 scripts/train_final.py --data data.xlsx --save-dir runs/final_model
!ls -la runs/final_model/

## Cell 4: Upload test organism FASTA files
Upload via Colab sidebar (Files → Upload):
- `Kareniaceae_fasta_for_targetp2.fsa`
- `Kareniaceae_fasta_for_targetp2_fsa.tab` (ASAFind output)
- (and similarly for other organisms: Bacillariophyta, Cryptophyceae, haptophyta, Prochlorococcus)

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload FASTA + .tab files

## Cell 5: Predict & Evaluate on Kareniaceae (example)
Repeat for each test organism by changing the filenames.

In [ ]:
ORGANISM = "Kareniaceae"  # Change this for each organism
FASTA = f"{ORGANISM}_fasta_for_targetp2.fsa"
TAB = f"{ORGANISM}_fasta_for_targetp2_fsa.tab"

# Step 1: Predict
!python3 scripts/predict.py \
    --model-dir runs/final_model \
    --fasta {FASTA} \
    --probs-out {ORGANISM}_probs.csv \
    > {ORGANISM}_predictions.txt

# Step 2: Compare with ASAFind
!python3 scripts/compare_predictions.py \
    --model-output {ORGANISM}_predictions.txt \
    --asafind-tab {TAB} \
    --out {ORGANISM}_comparison.csv

# Step 3: Full Baldi metrics
!echo "\n========== FULL METRICS =========="
!python3 scripts/compute_full_metrics.py --csv {ORGANISM}_comparison.csv

# Step 4: Prior correction
!echo "\n========== WITH PRIOR CORRECTION =========="
!python3 scripts/apply_prior_correction.py \
    --probs {ORGANISM}_probs.csv \
    --asafind-tab {TAB} \
    --target-prior-from-asafind \
    --out {ORGANISM}_corrected.csv

## Cell 6: Download results

In [ ]:
from google.colab import files
import glob

# Download all result files
for f in glob.glob("runs/cv_262seq/*") + glob.glob("runs/final_model/*") + glob.glob("*_comparison.csv") + glob.glob("*_corrected.csv"):
    print(f"Downloading {f}")
    files.download(f)